<a href="https://colab.research.google.com/github/Wangsamuels/argon/blob/main/eth_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ta
!pip install lightgbm

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29482 sha256=b0d8695e532675ffe74204b036865832c26483c64bcad2836d984b4c2205b466
  Stored in directory: /root/.cache/pip/wheels/e3/3a/ee/4955a26c90a4b7deb6d725dc8ec7b8604a7aef44e43a2e8af7
Successfully built ta


In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import lightgbm as lgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
import requests
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

PREDICTION_HORIZON_HOURS = 8
HORIZON_KEY = f'{PREDICTION_HORIZON_HOURS}h'

try:
    import talib
    TALIB_AVAILABLE = True
except ImportError:
    TALIB_AVAILABLE = False
    print("TA-Lib not available. Using manual technical indicators.")


# ---------------------------------------------------------------------------
# Loss
# ---------------------------------------------------------------------------

class ZPTAELoss:
    """
    Z Power-Tanh Absolute Error Loss Function
    As specified in Allora competition requirements
    """

    @staticmethod
    def power_tanh(x, alpha=0.25, beta=2):
        return x / (1 + np.abs(x)**beta)**((1 - alpha) / beta)

    @staticmethod
    def loss_zptae(y_true, y_pred, sigma, mean=0, alpha=0.25, beta=2,
                   gamma=4, penalty_norm=0.01):
        z_true = (y_true - mean) / sigma
        z_pred = (y_pred - mean) / sigma

        pt_true = ZPTAELoss.power_tanh(z_true, alpha=alpha, beta=beta)
        pt_pred = ZPTAELoss.power_tanh(z_pred, alpha=alpha, beta=beta)

        main_term = np.abs(pt_pred - pt_true)
        penalty_term = (penalty_norm * np.abs(z_pred - z_true))**gamma
        return main_term + penalty_term


# ---------------------------------------------------------------------------
# Data ingestion
# ---------------------------------------------------------------------------

class DataIngestion:
    """Handles both Tiingo (training) and DIA (current price) APIs."""

    def __init__(self, tiingo_api_key='f1ce7fb13ab049fff2477baf20c893b59b5add8e'):
        self.tiingo_api_key = tiingo_api_key
        self.tiingo_base_url = 'https://api.tiingo.com/tiingo/crypto/prices'
        self.dia_base_url = (
            'https://api.diadata.org/v1/assetQuotation/Ethereum/0x0000000000000000000000000000000000000000'
        )
        self.headers = {
            'Content-Type': 'application/json',
            'Authorization': f'Token {tiingo_api_key}'
        }
        self.data = None
        self.last_data_timestamp = None

    def fetch_current_price_from_dia(self):
        """Fetch current ETH price from DIA API."""
        try:
            print("Fetching current ETH price from DIA API...")
            response = requests.get(self.dia_base_url, timeout=10)
            if response.status_code != 200:
                print(f"DIA API request failed: {response.status_code}")
                return None, None

            data = response.json()
            current_price = data['Price']
            timestamp = pd.to_datetime(data['Time'])
            print(f"Current ETH price from DIA: ${current_price:.4f}")
            print(f"Price timestamp: {timestamp}")
            return current_price, timestamp

        except Exception as e:
            print(f"Error fetching DIA data: {e}")
            return None, None

    def fetch_eth_training_data(self, days_back=730, resample_freq='1hour'):
        """Fetch ETH/USDT historical training data from Tiingo API."""
        try:
            end_date = datetime.now()
            start_date = end_date - timedelta(days=days_back)
            start_str = start_date.strftime('%Y-%m-%d')
            end_str = end_date.strftime('%Y-%m-%d')

            print(f"Fetching ETH/USDT data from Tiingo ({start_str} to {end_str}, {resample_freq})...")

            params = {
                'tickers': 'ethusd',
                'startDate': start_str,
                'endDate': end_str,
                'resampleFreq': resample_freq
            }
            response = requests.get(
                self.tiingo_base_url, headers=self.headers, params=params
            )

            if response.status_code != 200:
                print(f"Tiingo API failed: {response.status_code}")
                print(response.text)
                return None

            data_json = response.json()
            if not data_json:
                print("No data returned from Tiingo API")
                return None

            eth_data = (data_json[0]['priceData']
                        if isinstance(data_json, list)
                        else data_json['priceData'])
            if not eth_data:
                print("No price data found for ETH/USDT")
                return None

            rows = [{
                'Date': pd.to_datetime(p['date']),
                'Open': p['open'],
                'High': p['high'],
                'Low': p['low'],
                'Close': p['close'],
                'Volume': p['volume']
            } for p in eth_data]

            data = pd.DataFrame(rows).set_index('Date').sort_index().dropna()
            if data.empty:
                raise ValueError("No valid data after cleaning")

            self.last_data_timestamp = data.index[-1]

            # Hourly log returns
            data['traditional_log_return'] = np.log(data['Close'] / data['Close'].shift(1))
            # Target: 8h-ahead log return
            data[f'log_return_{HORIZON_KEY}'] = (
                data['traditional_log_return'].shift(-PREDICTION_HORIZON_HOURS)
            )

            self.data = data.dropna()
            print(f"Loaded {len(self.data)} hourly bars from Tiingo")
            print(f"Range: {self.data.index.min()} → {self.data.index.max()}")
            print(f"Price: ${self.data['Close'].min():.2f} – ${self.data['Close'].max():.2f}")
            return self.data

        except Exception as e:
            print(f"Error fetching training data: {e}")
            return None


# ---------------------------------------------------------------------------
# Feature engineering
# ---------------------------------------------------------------------------

class FeatureEngineering:
    """Builds the feature matrix for LightGBM."""

    def __init__(self, lookback_windows=None):
        if lookback_windows is None:
            lookback_windows = [PREDICTION_HORIZON_HOURS, 12, 24, 48, 96, 168]
        self.lookback_windows = lookback_windows
        self.scaler = RobustScaler()
        self.feature_columns = []

    def create_features(self, data):
        df = data.copy()

        # --- Lagged returns ---
        for lag in sorted({1, 2, 3, 4, PREDICTION_HORIZON_HOURS, 12, 24, 48, 72, 96, 168}):
            df[f'ret_lag_{lag}'] = df['traditional_log_return'].shift(lag)

        # Horizon-aligned momentum
        for window in sorted({4, PREDICTION_HORIZON_HOURS, 12, 24, 48}):
            df[f'ret_sum_{window}h'] = df['traditional_log_return'].rolling(window).sum()
            df[f'ret_sum_sign_{window}h'] = np.sign(df[f'ret_sum_{window}h'])

        # --- Rolling volatility ---
        for window in self.lookback_windows:
            df[f'vol_std_{window}h'] = df['traditional_log_return'].rolling(window).std()
            df[f'vol_ewma_{window}h'] = (
                df['traditional_log_return'].ewm(halflife=window / 2).std()
            )
            df[f'realized_vol_{window}h'] = np.sqrt(
                df['traditional_log_return'].rolling(window).apply(lambda x: np.sum(x**2))
            )

        # --- Moving-average returns ---
        for window in self.lookback_windows:
            df[f'sma_ret_{window}h'] = df['traditional_log_return'].rolling(window).mean()
            df[f'ema_ret_{window}h'] = df['traditional_log_return'].ewm(span=window).mean()

        # --- Price-based technicals ---
        for window in [24, 48, 168]:
            df[f'price_sma_{window}h'] = df['Close'].rolling(window).mean()
            df[f'price_ratio_{window}h'] = df['Close'] / df[f'price_sma_{window}h']
            df[f'hl_ratio_{window}h'] = (
                df['High'].rolling(window).max() / df['Low'].rolling(window).min()
            )

        # --- Technical indicators ---
        if TALIB_AVAILABLE:
            df = self._add_talib_features(df)
        else:
            df = self._add_manual_technical_features(df)

        # --- Microstructure ---
        df['bid_ask_spread'] = (df['High'] - df['Low']) / df['Close']
        df['volume_price_trend'] = df['Volume'] * df['traditional_log_return']
        for window in [24, 48]:
            df[f'volume_sma_{window}h'] = df['Volume'].rolling(window).mean()
            df[f'volume_ratio_{window}h'] = df['Volume'] / df[f'volume_sma_{window}h']

        # --- Rankings ---
        df['ret_rank_24h'] = df['traditional_log_return'].rolling(24).rank(pct=True)
        df['vol_rank_24h'] = df['vol_std_24h'].rolling(168).rank(pct=True)

        # --- Regime ---
        df['high_vol_regime'] = (
            df['vol_std_24h'] > df['vol_std_24h'].rolling(168).quantile(0.75)
        ).astype(int)
        df['trend_strength'] = np.abs(df['sma_ret_24h'])

        # --- Time features ---
        df['hour'] = df.index.hour
        df['day_of_week'] = df.index.dayofweek
        df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)
        df['is_night'] = ((df.index.hour >= 22) | (df.index.hour <= 6)).astype(int)
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
        df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

        return df.dropna()

    def _add_manual_technical_features(self, df):
        for window in [14, 24]:
            delta = df['Close'].diff()
            gain = delta.where(delta > 0, 0).rolling(window).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window).mean()
            rs = gain / loss
            df[f'rsi_{window}h'] = 100 - (100 / (1 + rs))

        sma = df['Close'].rolling(48).mean()
        std = df['Close'].rolling(48).std()
        df['bb_upper'] = sma + 2 * std
        df['bb_lower'] = sma - 2 * std
        df['bb_position'] = (df['Close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])

        ema12 = df['Close'].ewm(span=12).mean()
        ema26 = df['Close'].ewm(span=26).mean()
        df['macd'] = ema12 - ema26
        df['macd_signal'] = df['macd'].ewm(span=9).mean()
        df['macd_histogram'] = df['macd'] - df['macd_signal']
        return df

    def _add_talib_features(self, df):
        c, h, lo = df['Close'].values, df['High'].values, df['Low'].values
        df['rsi_14h'] = talib.RSI(c, timeperiod=14)
        df['rsi_24h'] = talib.RSI(c, timeperiod=24)
        macd, sig, hist = talib.MACD(c)
        df['macd'], df['macd_signal'], df['macd_histogram'] = macd, sig, hist
        sk, sd = talib.STOCH(h, lo, c)
        df['stoch_k'], df['stoch_d'] = sk, sd
        return df

    def prepare_model_data(self, df, current_price, test_size=0.2):
        """Split into train/test with an embargo gap equal to the forecast horizon."""
        df = df.copy()

        target_col = f'target_modified_log_return_{PREDICTION_HORIZON_HOURS}h'
        df[target_col] = np.log(df['Close'].shift(-PREDICTION_HORIZON_HOURS) / df['Close'])

        # Select numeric feature columns, excluding raw OHLCV and the target
        exclude = {
            'traditional_log_return', f'log_return_{HORIZON_KEY}',
            'Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close',
            target_col,
        }
        feature_cols = sorted([
            c for c in df.columns
            if c not in exclude and df[c].dtype in ('int64', 'float64', 'int32', 'float32')
        ])
        self.feature_columns = feature_cols
        print(f"Selected {len(feature_cols)} features")

        df_clean = df[feature_cols + [target_col]].dropna()

        # Time-series split with embargo
        split_idx = int(len(df_clean) * (1 - test_size))
        train_end = max(split_idx - PREDICTION_HORIZON_HOURS, int(len(df_clean) * 0.5))

        X_train = df_clean[feature_cols].iloc[:train_end]
        X_test = df_clean[feature_cols].iloc[split_idx:]
        y_train = df_clean[target_col].iloc[:train_end]
        y_test = df_clean[target_col].iloc[split_idx:]

        # Scale
        X_train_s = pd.DataFrame(
            self.scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index
        )
        X_test_s = pd.DataFrame(
            self.scaler.transform(X_test), columns=feature_cols, index=X_test.index
        )

        print(f"Training samples: {len(X_train_s)}")
        print(f"Testing samples:  {len(X_test_s)}")
        return X_train_s, X_test_s, y_train, y_test, feature_cols, df_clean


# ---------------------------------------------------------------------------
# LightGBM model
# ---------------------------------------------------------------------------

class DirectionalLightGBM:
    """
    Direction classifier + magnitude regressor combined into a single
    predict() call that returns signed log-returns.
    """

    def __init__(self, classifier, regressor, feature_cols, threshold=0.5,
                 min_confidence=0.0, momentum_col=None, median_abs_return=0.01):
        self.classifier = classifier
        self.regressor = regressor
        self.feature_cols = list(feature_cols)
        self.threshold = float(threshold)
        self.min_confidence = float(min_confidence)
        self.momentum_col = momentum_col
        self.median_abs_return = median_abs_return
        self.feature_importances_ = getattr(classifier, 'feature_importances_', None)

    def _select_features(self, X):
        if hasattr(X, 'loc'):
            missing = [c for c in self.feature_cols if c not in X.columns]
            if missing:
                raise ValueError(f"Missing features: {missing[:8]}")
            return X[self.feature_cols]
        return X

    def _direction_from_proba(self, proba_up, momentum=None):
        lgb_dir = np.where(proba_up >= self.threshold, 1.0, -1.0)
        if momentum is None or self.min_confidence <= 0:
            return lgb_dir
        mom_dir = np.sign(np.asarray(momentum).ravel())
        mom_dir = np.where(mom_dir == 0, lgb_dir, mom_dir)
        confidence = np.abs(proba_up - self.threshold)
        return np.where(confidence >= self.min_confidence, lgb_dir, mom_dir)

    def predict(self, X):
        X_use = self._select_features(X)
        proba_up = self.classifier.predict_proba(X_use)[:, 1]

        momentum = None
        if (self.momentum_col and hasattr(X_use, 'columns')
                and self.momentum_col in X_use.columns):
            momentum = X_use[self.momentum_col].values

        direction = self._direction_from_proba(proba_up, momentum)
        magnitude = np.maximum(np.abs(self.regressor.predict(X_use)), 1e-8)
        return direction * magnitude


class LightGBMTrainer:
    """Trains the DirectionalLightGBM model for 8h log-return prediction."""

    def __init__(self):
        self.model = None              # DirectionalLightGBM instance
        self.feature_importance = {}

    # -- helper eval metrics for early stopping --

    @staticmethod
    def dir_acc_eval(y_true_or_preds, y_pred_or_dataset):
        if hasattr(y_pred_or_dataset, 'get_label'):
            y_pred = np.asarray(y_true_or_preds).ravel()
            y_true = np.asarray(y_pred_or_dataset.get_label()).ravel()
        else:
            y_true = np.asarray(y_true_or_preds).ravel()
            y_pred = np.asarray(y_pred_or_dataset).ravel()

        if y_pred.min() >= 0.0 and y_pred.max() <= 1.0:
            pred_up = y_pred >= 0.5
        else:
            pred_up = y_pred >= 0.0

        true_up = (y_true >= 0.5) if set(np.unique(y_true)).issubset({0, 1}) else (y_true > 0)
        return 'dir_acc', float(np.mean(pred_up == true_up)), True

    # -- internal helpers --

    @staticmethod
    def _stationary_feature_cols(feature_cols):
        drop_prefixes = ('price_sma_', 'volume_sma_')
        drop_exact = {'bb_upper', 'bb_lower', 'macd', 'macd_signal',
                      'macd_histogram', 'hour', 'day_of_week'}
        selected = [
            c for c in feature_cols
            if c not in drop_exact and not c.startswith(drop_prefixes)
        ]
        return selected or list(feature_cols)

    @staticmethod
    def _recency_weights(y, half_life_hours=14 * 24):
        n = len(y)
        ages = np.arange(n - 1, -1, -1, dtype=float)
        recency = np.exp(-np.log(2.0) / max(half_life_hours, 1) * ages)
        abs_y = np.abs(np.asarray(y, dtype=float).ravel())
        typical = max(np.median(abs_y), 1e-8)
        move_w = 1.0 + abs_y / typical
        noise_cut = np.quantile(abs_y, 0.40) if len(abs_y) else 0.0
        noise_w = np.where(abs_y >= noise_cut, 1.0, 0.15)
        w = recency * move_w * noise_w
        return w / np.mean(w)

    @staticmethod
    def _best_iteration(model, default=200):
        for attr in ('best_iteration_', 'best_iteration'):
            val = getattr(model, attr, None)
            if val is not None and int(val) > 0:
                return int(val)
        booster = getattr(model, 'booster_', None)
        if booster is not None:
            val = getattr(booster, 'best_iteration', None)
            if val is not None and int(val) > 0:
                return int(val)
        return default

    @staticmethod
    def _calibrate_threshold(proba, y_true, momentum=None):
        y_true = np.asarray(y_true).ravel()
        proba = np.asarray(proba).ravel()
        true_up = y_true > 0
        if momentum is None:
            momentum = np.zeros_like(proba)
        mom_dir = np.sign(np.asarray(momentum).ravel())

        best = {'acc': -1.0, 'threshold': 0.5, 'min_confidence': 0.0}
        for threshold in np.linspace(0.40, 0.60, 21):
            lgb_dir = np.where(proba >= threshold, 1.0, -1.0)
            confidence = np.abs(proba - threshold)
            for min_conf in (0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12):
                blended = np.where(
                    (confidence >= min_conf) | (mom_dir == 0),
                    lgb_dir,
                    np.where(mom_dir == 0, lgb_dir, mom_dir)
                )
                acc = float(np.mean((blended > 0) == true_up))
                if acc > best['acc']:
                    best = {'acc': acc, 'threshold': float(threshold),
                            'min_confidence': float(min_conf)}
        return best

    def _recent_slice(self, X, y, max_hours=150 * 24):
        if len(X) <= max_hours:
            return X, y
        print(f"Using most recent {max_hours / 24:.0f} days for training")
        return X.iloc[-max_hours:], y.iloc[-max_hours:]

    def _purged_split(self, X, y, val_size=0.15):
        val_split = int(len(X) * (1 - val_size))
        embargo = PREDICTION_HORIZON_HOURS
        train_end = max(val_split - embargo, int(len(X) * 0.5))
        if train_end < 50 or (len(X) - val_split) < 20:
            return X, y, X.iloc[-min(50, len(X)):], y.iloc[-min(50, len(y)):]
        return X.iloc[:train_end], y.iloc[:train_end], X.iloc[val_split:], y.iloc[val_split:]

    def _fit_with_early_stopping(self, estimator, X_tr, y_tr, X_val, y_val,
                                 sample_weight, eval_metric):
        try:
            estimator.fit(
                X_tr, y_tr, sample_weight=sample_weight,
                eval_set=[(X_val, y_val)], eval_metric=eval_metric,
                callbacks=[lgb.early_stopping(80, first_metric_only=True, verbose=50)]
            )
        except TypeError:
            estimator.fit(
                X_tr, y_tr, sample_weight=sample_weight,
                eval_set=[(X_val, y_val)], eval_metric=eval_metric,
                early_stopping_rounds=80, verbose=50
            )
        return estimator

    # -- public API --

    def train(self, X_train, y_train, feature_cols=None, val_size=0.18):
        """
        Train direction classifier + magnitude regressor, calibrate decision
        threshold, retrain on full recent window, and store the final model.
        """
        if feature_cols is None:
            feature_cols = list(X_train.columns)

        feature_cols = self._stationary_feature_cols(feature_cols)
        momentum_col = f'ret_sum_{PREDICTION_HORIZON_HOURS}h'
        if momentum_col not in feature_cols and momentum_col in X_train.columns:
            feature_cols = list(feature_cols) + [momentum_col]

        X_recent, y_recent = self._recent_slice(X_train[feature_cols], y_train)
        X_tr, y_tr, X_val, y_val = self._purged_split(X_recent, y_recent, val_size)

        shared_params = dict(
            learning_rate=0.06,
            max_depth=4,
            num_leaves=24,
            n_estimators=2500,
            subsample=0.7,
            colsample_bytree=0.5,
            reg_alpha=0.8,
            reg_lambda=6.0,
            min_child_samples=80,
            subsample_freq=1,
            min_split_gain=0.01,
            random_state=42,
            verbose=-1,
        )

        sample_w = self._recency_weights(y_tr)
        y_tr_dir = (y_tr > 0).astype(int)
        y_val_dir = (y_val > 0).astype(int)
        val_momentum = X_val[momentum_col].values if momentum_col in X_val.columns else None

        print(f"Training LightGBM  (features={len(feature_cols)}, "
              f"train={len(X_tr)}, val={len(X_val)})")

        # ---- Direction classifier ----
        clf = lgb.LGBMClassifier(
            objective='binary', metric='None', class_weight='balanced', **shared_params
        )
        clf = self._fit_with_early_stopping(
            clf, X_tr, y_tr_dir, X_val, y_val_dir, sample_w, self.dir_acc_eval
        )

        n_clf = max(self._best_iteration(clf, 200), 40)
        print(f"Retraining classifier on full recent window ({n_clf} trees)")
        final_clf = lgb.LGBMClassifier(
            objective='binary', metric='None', class_weight='balanced',
            **{**shared_params, 'n_estimators': n_clf}
        )
        recent_w = self._recency_weights(y_recent)
        final_clf.fit(X_recent, (y_recent > 0).astype(int), sample_weight=recent_w)

        # Calibrate threshold
        val_proba = final_clf.predict_proba(X_val)[:, 1]
        rule = self._calibrate_threshold(val_proba, y_val.values, val_momentum)
        print(f"Calibrated rule: threshold={rule['threshold']:.3f}, "
              f"min_confidence={rule['min_confidence']:.3f}, val DA={rule['acc']:.4f}")

        # ---- Magnitude regressor ----
        reg = lgb.LGBMRegressor(objective='regression_l1', metric='None', **shared_params)
        reg = self._fit_with_early_stopping(
            reg, X_tr, y_tr, X_val, y_val, sample_w, self.dir_acc_eval
        )

        n_reg = max(self._best_iteration(reg, 200), 40)
        final_reg = lgb.LGBMRegressor(
            objective='regression_l1', metric='None',
            **{**shared_params, 'n_estimators': n_reg}
        )
        final_reg.fit(X_recent, y_recent, sample_weight=recent_w)

        # ---- Compose final model ----
        self.model = DirectionalLightGBM(
            classifier=final_clf,
            regressor=final_reg,
            feature_cols=feature_cols,
            threshold=rule['threshold'],
            min_confidence=rule['min_confidence'],
            momentum_col=momentum_col if momentum_col in feature_cols else None,
            median_abs_return=float(np.median(np.abs(y_recent))),
        )

        val_preds = self.model.predict(X_val)
        val_da = float(np.mean(np.sign(val_preds) == np.sign(y_val.values)))
        print(f"LightGBM validation directional accuracy: {val_da:.4f}")

        if self.model.feature_importances_ is not None:
            self.feature_importance = dict(zip(feature_cols, self.model.feature_importances_))

        print("LightGBM training completed")
        return self.model

    def predict(self, X):
        """Predict signed 8h log-returns."""
        if self.model is None:
            raise RuntimeError("Model not trained yet — call train() first")
        return self.model.predict(X)


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------

class ModelEvaluation:

    def __init__(self):
        self.results = {}
        self.zptae = ZPTAELoss()

    @staticmethod
    def directional_accuracy(y_true, y_pred, threshold=0.0):
        y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
        n = min(len(y_true), len(y_pred))
        y_true, y_pred = y_true[-n:], y_pred[-n:]
        if threshold > 0:
            mask = np.abs(y_true) > threshold
            if mask.sum() == 0:
                return 0.0
            y_true, y_pred = y_true[mask], y_pred[mask]
        return float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    def evaluate(self, trainer, X_test, y_test, current_price):
        """Evaluate the trained LightGBM model on the test set."""
        print("\nEvaluating LightGBM model...")
        y_pred = trainer.predict(X_test)

        n = min(len(y_test), len(y_pred))
        y_true_a = y_test.iloc[-n:] if hasattr(y_test, 'iloc') else y_test[-n:]
        y_pred_a = y_pred[-n:]

        sigma = y_true_a.tail(min(100, len(y_true_a))).std()

        rmse = np.sqrt(mean_squared_error(y_true_a, y_pred_a))
        mae = mean_absolute_error(y_true_a, y_pred_a)
        zptae_val = float(np.mean(self.zptae.loss_zptae(y_true_a, y_pred_a, sigma)))
        da = self.directional_accuracy(y_true_a, y_pred_a)
        da_sig = self.directional_accuracy(y_true_a, y_pred_a, threshold=0.02)
        ic = np.corrcoef(y_true_a, y_pred_a)[0, 1] if n > 1 else 0.0

        # Price-level metrics
        pred_prices = current_price * np.exp(y_pred_a)
        actual_prices = current_price * np.exp(y_true_a)
        price_rmse = np.sqrt(np.mean((pred_prices - actual_prices)**2))
        price_mae = np.mean(np.abs(pred_prices - actual_prices))

        metrics = {
            'rmse': rmse, 'mae': mae, 'zptae': zptae_val,
            'directional_accuracy': da,
            'directional_accuracy_significant': da_sig,
            'information_coefficient': float(ic),
            'price_rmse': price_rmse, 'price_mae': price_mae,
            'price_rmse_pct': price_rmse / current_price * 100,
            'price_mae_pct': price_mae / current_price * 100,
            'n_predictions': n,
        }
        self.results = metrics

        print(f"  RMSE={rmse:.6f}  MAE={mae:.6f}  ZPTAE={zptae_val:.6f}")
        print(f"  Directional accuracy={da:.4f}  (|ret|>0.02: {da_sig:.4f})")
        print(f"  IC={ic:.4f}")
        return metrics

    def report_directional_accuracy(self, trainer, X_train, y_train,
                                    X_test, y_test, current_price):
        train_pred = trainer.predict(X_train)
        test_pred = trainer.predict(X_test)

        train_da = self.directional_accuracy(y_train.values, train_pred)
        test_da = self.directional_accuracy(y_test.values, test_pred)
        train_da_sig = self.directional_accuracy(y_train.values, train_pred, 0.02)
        test_da_sig = self.directional_accuracy(y_test.values, test_pred, 0.02)

        print(f"\n{'='*60}")
        print(f"DIRECTIONAL ACCURACY ({HORIZON_KEY})")
        print(f"{'='*60}")
        print(f"  Train: {train_da:.4f}   (|ret|>0.02: {train_da_sig:.4f})")
        print(f"  Test:  {test_da:.4f}   (|ret|>0.02: {test_da_sig:.4f})")
        print(f"{'='*60}")

        return {
            'train': train_da, 'test': test_da,
            'train_significant': train_da_sig, 'test_significant': test_da_sig,
        }


# ---------------------------------------------------------------------------
# Production pipeline
# ---------------------------------------------------------------------------

class ProductionPipeline:
    """End-to-end pipeline: ingest → features → train → evaluate → predict."""

    def __init__(self, tiingo_api_key='f1ce7fb13ab049fff2477baf20c893b59b5add8e'):
        self.data_ingestion = DataIngestion(tiingo_api_key=tiingo_api_key)
        self.feature_engineering = FeatureEngineering()
        self.trainer = LightGBMTrainer()
        self.evaluator = ModelEvaluation()

    def full_pipeline(self, days_back=730, test_size=0.2, resample_freq='1hour'):
        print("=" * 80)
        print("ETH/USDT 8h LOG-RETURN PREDICTION PIPELINE  (LightGBM)")
        print("Training: Tiingo API  |  Current Price: DIA API")
        print(f"Target: ln(new_price / current_price) at {HORIZON_KEY}")
        print()

        # 1. Current price
        print("1. Fetching current price from DIA...")
        current_price, current_ts = self.data_ingestion.fetch_current_price_from_dia()
        if current_price is None:
            raise ValueError("Failed to fetch current price from DIA API")

        # 2. Training data
        print("\n2. Fetching training data from Tiingo...")
        training_data = self.data_ingestion.fetch_eth_training_data(days_back, resample_freq)
        if training_data is None:
            raise ValueError("Failed to fetch training data from Tiingo API")

        last_ts = training_data.index[-1]
        gap_hours = (current_ts - last_ts).total_seconds() / 3600
        print(f"Training data ends: {last_ts}  |  Gap: {gap_hours:.1f}h")
        if gap_hours > 24:
            print("⚠ Large gap between training data and current time")

        # 3. Features
        print("\n3. Feature engineering...")
        featured = self.feature_engineering.create_features(training_data)

        # 4. Train/test split
        print("\n4. Preparing train/test split...")
        X_train, X_test, y_train, y_test, feat_cols, df_clean = \
            self.feature_engineering.prepare_model_data(featured, current_price, test_size)

        # 5. Train
        print("\n5. Training LightGBM...")
        self.trainer.train(X_train, y_train, feature_cols=feat_cols)

        # 6. Evaluate
        print("\n6. Evaluating...")
        metrics = self.evaluator.evaluate(self.trainer, X_test, y_test, current_price)
        da_report = self.evaluator.report_directional_accuracy(
            self.trainer, X_train, y_train, X_test, y_test, current_price
        )

        if metrics['directional_accuracy'] < 0.50:
            print("⚠ Test directional accuracy is below 50%")

        return {
            'metrics': metrics,
            'directional_accuracy': da_report,
            'current_price': current_price,
            'current_timestamp': current_ts,
            'training_data_end': last_ts,
            'gap_hours': gap_hours,
            'feature_count': len(feat_cols),
            'total_training_points': len(df_clean),
        }

    def generate_prediction(self, use_real_time=True):
        """Generate a single 8h ETH log-return prediction."""
        print(f"\n{'='*60}")
        print(f"GENERATING {HORIZON_KEY} ETH/USDT LOG-RETURN PREDICTION")
        print(f"{'='*60}")

        current_price, current_ts = self.data_ingestion.fetch_current_price_from_dia()
        if current_price is None:
            raise ValueError("Failed to fetch current price from DIA API")

        if use_real_time:
            recent_data = self.data_ingestion.fetch_eth_training_data(60, '1hour')
        else:
            if self.data_ingestion.data is None:
                raise ValueError("No data — run full_pipeline() first or set use_real_time=True")
            recent_data = self.data_ingestion.data

        if self.trainer.model is None:
            raise ValueError("No trained model — run full_pipeline() first")

        featured = self.feature_engineering.create_features(recent_data)
        latest = featured.iloc[[-1]]

        # Align to the trained feature set
        available = [c for c in self.feature_engineering.feature_columns if c in latest.columns]
        X = latest[available].copy()
        for c in self.feature_engineering.feature_columns:
            if c not in X.columns:
                X[c] = 0.0
        X = X[self.feature_engineering.feature_columns]

        X_scaled = pd.DataFrame(
            self.feature_engineering.scaler.transform(X),
            columns=X.columns, index=X.index,
        )

        log_return = self.trainer.predict(X_scaled)[0]
        pct_change = (np.exp(log_return) - 1) * 100
        recent_vol = recent_data['traditional_log_return'].tail(100).std()

        print(f"\nTimestamp:     {current_ts}")
        print(f"Horizon:      {HORIZON_KEY}")
        print(f"Log return:   {log_return:+.6f}")
        print(f"Change:       {pct_change:+.2f}%")
        print(f"Volatility (100h): {recent_vol:.4f}")

        return {
            'timestamp': current_ts,
            'horizon': HORIZON_KEY,
            'log_return': log_return,
            'percentage_change': pct_change,
            'recent_volatility': recent_vol,
        }

    def monitor_api_status(self):
        print("\nMonitoring API Status...")
        print("-" * 40)
        price, ts = self.data_ingestion.fetch_current_price_from_dia()
        if price:
            print(f"DIA API:    OK  – ETH ${price:.4f} at {ts}")
        else:
            print("DIA API:    Not responding")

        try:
            d = self.data_ingestion.fetch_eth_training_data(7, '1hour')
            if d is not None and len(d):
                print(f"Tiingo API: OK  – {len(d)} bars, {d.index.min()} → {d.index.max()}")
            else:
                print("Tiingo API: No data returned")
        except Exception as e:
            print(f"Tiingo API: Error – {e}")
        print("-" * 40)


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

def main():
    pipeline = ProductionPipeline(
        tiingo_api_key='f1ce7fb13ab049fff2477baf20c893b59b5add8e'
    )

    pipeline.monitor_api_status()

    print("\nStarting full pipeline...")
    results = pipeline.full_pipeline(days_back=365, test_size=0.15, resample_freq='1hour')

    prediction = pipeline.generate_prediction(use_real_time=True)

    print(f"\n{'='*60}")
    print(f"Pipeline complete")
    print(f"  {HORIZON_KEY} log return: {prediction['log_return']:+.6f} "
          f"({prediction['percentage_change']:+.2f}%)")
    da = results['directional_accuracy']
    print(f"  DA  train={da['train']:.4f}  test={da['test']:.4f}")
    print(f"{'='*60}")

    return {'pipeline': pipeline, 'results': results, 'prediction': prediction}


if __name__ == "__main__":
    print("ETH/USDT 8h Log-Return Prediction System (LightGBM)")
    output = main()


TA-Lib not available. Using manual technical indicators.
ETH/USDT 8h Log-Return Prediction System (LightGBM)

Monitoring API Status...
----------------------------------------
Fetching current ETH price from DIA API...
Current ETH price from DIA: $2714.1061
Price timestamp: 2026-09-29 11:01:59+00:00
DIA API:    OK  – ETH $2714.1061 at 2026-09-29 11:01:59+00:00
Fetching ETH/USDT data from Tiingo (2026-09-22 to 2026-09-29, 1hour)...
Loaded 160 hourly bars from Tiingo
Range: 2026-09-22 01:00:00+00:00 → 2026-09-28 16:00:00+00:00
Price: $2640.62 – $2779.40
Tiingo API: OK  – 160 bars, 2026-09-22 01:00:00+00:00 → 2026-09-28 16:00:00+00:00
----------------------------------------

Starting full pipeline...
ETH/USDT 8h LOG-RETURN PREDICTION PIPELINE  (LightGBM)
Training: Tiingo API  |  Current Price: DIA API
Target: ln(new_price / current_price) at 8h

1. Fetching current price from DIA...
Fetching current ETH price from DIA API...
Current ETH price from DIA: $2714.1061
Price timestamp: 2026-09